# O-Level Mathematics Regression — EDA
Use this notebook to inspect the raw dataset and validate the cleaning decisions used by the training pipeline.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data_preparation import DataPreparation

with open(ROOT / 'src/config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)
raw = pd.read_csv(ROOT / config['data_path'])
raw.shape


In [ ]:
raw.head()


In [ ]:
summary = pd.DataFrame({
    'dtype': raw.dtypes.astype(str),
    'missing': raw.isna().sum(),
    'missing_pct': (raw.isna().mean() * 100).round(2),
    'unique': raw.nunique(dropna=True),
})
summary.sort_values('missing_pct', ascending=False)


In [ ]:
print('Duplicate rows:', raw.duplicated().sum())
for col in ['direct_admission', 'CCA', 'learning_style', 'gender', 'tuition', 'mode_of_transport', 'bag_color']:
    if col in raw.columns:
        print(f'\n{col}:', raw[col].value_counts(dropna=False).head(20).to_dict())


In [ ]:
prep = DataPreparation(config)
clean = prep.clean_data(raw)
clean.describe(include='all').T


In [ ]:
target = config['target_column']
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(clean[target], bins=25, kde=True, ax=ax)
ax.set_title('Distribution of final mathematics scores')
plt.show()


In [ ]:
numeric = clean.select_dtypes(include='number')
corr = numeric.corr(numeric_only=True)[target].sort_values(ascending=False)
corr


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(numeric.corr(numeric_only=True), cmap='coolwarm', center=0, ax=ax)
ax.set_title('Numerical feature correlation heatmap')
plt.show()
